In [ ]:
# /// script
# requires-python = ">=3.11"
# dependencies = [
#     "neurocnl",
#     "snntorch",
#     "tonic",
# ]
# ///

# NeuroMorphic Pipeline — Snntorch Sim

Generated 2026-07-04 00:00 UTC.

**Architecture:** defined in the Architecture tab (CNL spec below).
**Pipeline config:** edit `config` in the next cell to change training parameters.

In [ ]:
# ── Pipeline configuration ───────────────────────────────────────────
# Workspace settings used when this notebook was generated.

config = {
    "dataset":   "NMNIST",
    "framework": "snntorch_sim",
}

print('Config loaded:', config)

In [ ]:
import json as _json


def _nmtk_emit(
    epoch: int, total: int, loss: float, accuracy: float, layer_rates: dict
) -> None:
    print(
        _json.dumps(
            {
                "__nmtk_progress__": True,
                "epoch": epoch,
                "total_epochs": total,
                "loss": loss,
                "accuracy": accuracy,
                "layer_spike_rates": layer_rates,
            }
        ),
        flush=True,
    )


In [ ]:
import torch
from torch.utils.data import DataLoader
try:
    import tonic
    import tonic.transforms as _tf
    _sensor_size = tonic.datasets.NMNIST.sensor_size
    _frame_tf = _tf.ToFrame(sensor_size=_sensor_size, n_time_bins=25)  # ponytail: fixed bins so all samples have same T; matches num_steps
    train_ds = tonic.datasets.NMNIST(save_to='data/', train=True,  transform=_frame_tf)
    test_ds  = tonic.datasets.NMNIST(save_to='data/', train=False, transform=_frame_tf)
except ImportError:
    raise ImportError('pip install tonic')
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0)
print(f'N-MNIST: {len(train_loader.dataset)} train / {len(test_loader.dataset)} test samples')

## Architecture

Network compiled from CNL spec via NIR.

In [ ]:
# CNL spec — auto-generated from the Architecture canvas tab.
# To change the network, edit the Architecture tab and regenerate.
cnl_spec = '''
Define a network named graph.
# Network with 1 input, 13 hidden nodes, 1 output.
# flow: input → 0 → 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → 12 → output

# Layers:
Define a 2D convolution layer named 0 with weight kernel shape (16, 2, 5, 5), bias vector shape (16,), stride (2, 2), padding (1, 1), dilation (1, 1), groups 1, and input height and width (34, 34).
Define an integrate-and-fire neuron named 1 with resistance shape (16, 16, 16) and firing threshold shape (16, 16, 16).
Define an integrate-and-fire neuron named 10 with resistance shape (256,) and firing threshold shape (256,).
Define an affine transformation named 11 with weight matrix shape (10, 256) and bias vector shape (10,).
Define an integrate-and-fire neuron named 12 with resistance shape (10,) and firing threshold shape (10,).
Define a 2D convolution layer named 2 with weight kernel shape (16, 16, 3, 3), bias vector shape (16,), stride (1, 1), padding (1, 1), dilation (1, 1), groups 1, and input height and width (16, 16).
Define an integrate-and-fire neuron named 3 with resistance shape (16, 16, 16) and firing threshold shape (16, 16, 16).
Define a 2D sum pooling layer named 4 with kernel size (2, 2), stride (2, 2), and padding (0, 0).
Define a 2D convolution layer named 5 with weight kernel shape (8, 16, 3, 3), bias vector shape (8,), stride (1, 1), padding (1, 1), dilation (1, 1), groups 1, and input height and width (8, 8).
Define an integrate-and-fire neuron named 6 with resistance shape (8, 8, 8) and firing threshold shape (8, 8, 8).
Define a 2D sum pooling layer named 7 with kernel size (2, 2), stride (2, 2), and padding (0, 0).
Define a flatten layer named 8 with start dimension 0 and end dimension -1.
Define an affine transformation named 9 with weight matrix shape (256, 128) and bias vector shape (256,).
Define an input port named input with shape (2, 34, 34).
Define an output port named output with shape (10,).

# Connections:
input connects to 0.
0 connects to 1.
1 connects to 2.
2 connects to 3.
3 connects to 4.
4 connects to 5.
5 connects to 6.
6 connects to 7.
7 connects to 8.
8 connects to 9.
9 connects to 10.
10 connects to 11.
11 connects to 12.
12 connects to output.

'''

from neurocnl.compile import compile_to_nir

graph = compile_to_nir(cnl_spec)
print(f'Network: {len(graph.nodes)} nodes, {len(graph.edges)} edges')

In [ ]:
"""snnTorch network — auto-generated from NIR graph."""

import torch
import torch.nn as nn
import snntorch as snn
import numpy as np

_w = np.load('weights.npz')  # weights file saved alongside this notebook


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Conv2d layer: '0'  shape (16, 2, 5, 5)
        self.n_0 = nn.Conv2d(2, 16, (5, 5), stride=(2, 2), padding=(1, 1), bias=True)
        self.n_0.weight.data = torch.from_numpy(_w['n_0_weight'].copy())
        self.n_0.bias.data = torch.from_numpy(_w['n_0_bias'].copy())
        # IF population: '1'
        self.n_1 = snn.Leaky(beta=0.900000, threshold=1.0000, init_hidden=True, reset_delay=False)
        # IF population: '10'
        self.n_10 = snn.Leaky(beta=0.900000, threshold=1.0000, init_hidden=True, reset_delay=False)
        # Linear layer: '11'  shape (10, 256)
        self.n_11 = nn.Linear(256, 10, bias=True)
        self.n_11.weight.data = torch.from_numpy(_w['n_11_weight'].copy())
        self.n_11.bias.data = torch.from_numpy(_w['n_11_bias'].copy())
        # IF population: '12'
        self.n_12 = snn.Leaky(beta=0.900000, threshold=1.0000, init_hidden=True, reset_delay=False)
        # Conv2d layer: '2'  shape (16, 16, 3, 3)
        self.n_2 = nn.Conv2d(16, 16, (3, 3), stride=(1, 1), padding=(1, 1), bias=True)
        self.n_2.weight.data = torch.from_numpy(_w['n_2_weight'].copy())
        self.n_2.bias.data = torch.from_numpy(_w['n_2_bias'].copy())
        # IF population: '3'
        self.n_3 = snn.Leaky(beta=0.900000, threshold=1.0000, init_hidden=True, reset_delay=False)
        # SumPool2d: '4' — true sum via divisor_override=1
        self.n_4 = nn.AvgPool2d(kernel_size=(2, 2), stride=(2, 2), divisor_override=1)
        # Conv2d layer: '5'  shape (8, 16, 3, 3)
        self.n_5 = nn.Conv2d(16, 8, (3, 3), stride=(1, 1), padding=(1, 1), bias=True)
        self.n_5.weight.data = torch.from_numpy(_w['n_5_weight'].copy())
        self.n_5.bias.data = torch.from_numpy(_w['n_5_bias'].copy())
        # IF population: '6'
        self.n_6 = snn.Leaky(beta=0.900000, threshold=1.0000, init_hidden=True, reset_delay=False)
        # SumPool2d: '7' — true sum via divisor_override=1
        self.n_7 = nn.AvgPool2d(kernel_size=(2, 2), stride=(2, 2), divisor_override=1)
        # Flatten layer: '8'
        self.n_8 = nn.Flatten(start_dim=1, end_dim=-1)
        # Linear layer: '9'  shape (256, 128)
        self.n_9 = nn.Linear(128, 256, bias=True)
        self.n_9.weight.data = torch.from_numpy(_w['n_9_weight'].copy())
        self.n_9.bias.data = torch.from_numpy(_w['n_9_bias'].copy())

    def forward(self, x):
        # initialise hidden states
        self.n_1.init_leaky()
        self.n_10.init_leaky()
        self.n_12.init_leaky()
        self.n_3.init_leaky()
        self.n_6.init_leaky()
        # x: (T,B,C,H,W) time-first from tonic, or (B,C,H,W) for a single frame
        if x.dim() == 4:
            x = x.unsqueeze(0)  # (B,C,H,W) → (1,B,C,H,W)
        _x_seq = x
        spk_rec = []
        for t in range(_x_seq.shape[0]):
            x = _x_seq[t]
            x = self.n_8(x)
            x = self.n_11(x)
            x = self.n_5(x)
            x = self.n_0(x)
            x = self.n_9(x)
            spk_n_12 = self.n_12(x)
            x = spk_n_12
            x = self.n_4(x)
            spk_n_3 = self.n_3(x)
            x = spk_n_3
            x = self.n_2(x)
            spk_n_6 = self.n_6(x)
            x = spk_n_6
            spk_n_1 = self.n_1(x)
            x = spk_n_1
            x = self.n_7(x)
            spk_n_10 = self.n_10(x)
            x = spk_n_10
            spk_rec.append(x)
        return torch.stack(spk_rec, dim=0), x  # (T, batch, out), last spk


net = Net().float()  # ponytail: npz weights load as float64; cast to match DataLoader float32 input
print(f'Net: {sum(p.numel() for p in net.parameters())} parameters')

> **⚠️ UNSUPPORTED support for `snntorch_sim`**
>
> snnTorch simulator: nir.SumPool2d is not supported and cannot be executed. Remove or replace this node type before running the simulation.

In [ ]:
# ── Download as Python script ──────────────────────────────────
# Run this cell to download the notebook as a .py script.
import subprocess
subprocess.run(['jupyter', 'nbconvert', '--to', 'script',
                '__file__'], check=False)
print('Conversion triggered — check the file listing.')